# Payphone merge on Kaggle (Hugging Face checkpoint only)

This notebook does **one job on Kaggle**: load Qwen2.5-7B-Instruct in 4-bit, merge your LoRA, save a **standard Hugging Face folder**, zip it, and download **`payphone-merged-hf.zip`**.

**GGUF for Ollama is a separate step** (smaller surface area = fewer Kaggle failures):

1. **Colab (one-shot GGUF):** [colab_full_import.ipynb](https://colab.research.google.com/github/s4ifk99/ProjectPayphone/blob/main/notebooks/colab_full_import.ipynb) with your **LoRA adapter zip** from training → download `payphone-story.gguf`. (That notebook expects the adapter zip, not this merged-HF zip.)
2. **Local:** Unzip the checkpoint, clone [llama.cpp](https://github.com/ggerganov/llama.cpp), run `convert_hf_to_gguf.py`. See [IMPORT_OLLAMA.md](https://github.com/s4ifk99/ProjectPayphone/blob/main/IMPORT_OLLAMA.md) and repo [scripts/convert_to_gguf.sh](https://github.com/s4ifk99/ProjectPayphone/blob/main/scripts/convert_to_gguf.sh).

**One-shot merge + GGUF in the cloud** remains **Colab T4** + `colab_full_import.ipynb`.

---

**Do not open `colab_full_import.ipynb` on Kaggle.** That notebook uses `google.colab.files` and `/content/` — **those do not exist on Kaggle**, so upload never works. Use **this** notebook only (`kaggle_full_import.ipynb`).

**LoRA data:** In the editor, **Add data** → attach your dataset (e.g. **Storyteller lora**). It mounts read-only under **`/kaggle/input/<dataset-folder>/`** (folder name may differ from the title). Section 2 scans that tree automatically — no upload button.

**Setup**

1. [kaggle.com](https://kaggle.com) — **Settings** → **Internet** ON.
2. **Accelerator** → **GPU** (P100 or T4).
3. **Add data** → your LoRA dataset (folder or zip with `adapter_config.json` + `adapter_model.safetensors`).
4. Run all cells.

**Noise:** cuFFT/cuDNN “already registered” stderr is common when TF + PyTorch share CUDA; if checkpoint shards reach **100%**, continue. **Session → Restart** if OOM.


## 1. Install dependencies


In [ ]:
# Reduce CUDA fragmentation (run first; on Kaggle: Session → Restart if you hit GPU OOM)
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# Do NOT force numpy 2.x — Kaggle's tensorflow/numba expect older numpy; pip may still print red conflicts.
# If you see "numpy.dtype size changed": !pip install -q "numpy>=1.26,<2.1"

import torch
with open("/tmp/constraints.txt", "w") as f:
    f.write(f"torch=={torch.__version__}\n")
import numpy as np
print(f"torch {torch.__version__} (CUDA: {torch.cuda.is_available()}), numpy {np.__version__}")

!pip install -q -c /tmp/constraints.txt peft bitsandbytes "transformers==4.56.2" "accelerate==1.4.0"

import transformers
import peft
import bitsandbytes  # noqa: F401
print("OK:", "transformers", transformers.__version__, "peft", getattr(peft, "__version__", "?"))


## 2. Locate LoRA adapter (from Add data — not Colab upload)

After you attach **Storyteller lora** (or any LoRA dataset), run the next cell. It prints what is under `/kaggle/input` and finds `adapter_config.json`.


In [ ]:
import os
import zipfile

INPUT_ROOT = "/kaggle/input"
WORKING = "/kaggle/working"

if os.path.isdir(INPUT_ROOT):
    print("Mounted input datasets:", sorted(os.listdir(INPUT_ROOT)))
else:
    print("WARNING: /kaggle/input missing — use Add data to attach your LoRA dataset.")

def has_adapter(p):
    return os.path.exists(os.path.join(p, "adapter_config.json")) and os.path.exists(os.path.join(p, "adapter_model.safetensors"))

# First: recursive search for adapter_config.json anywhere under /kaggle/input
adapter_path = None
for root, dirs, files in os.walk(INPUT_ROOT):
    if "adapter_config.json" in files and "adapter_model.safetensors" in files:
        adapter_path = root
        print(f"Found adapter at: {adapter_path}")
        break

# If not found, try extracting zips
if not adapter_path:
    print("Scanning datasets...")
    for name in sorted(os.listdir(INPUT_ROOT)):
        d = os.path.join(INPUT_ROOT, name)
        if not os.path.isdir(d):
            continue
        contents = os.listdir(d)
        print(f"  {name}: {contents}")
        for f in contents:
            if f.endswith(".zip"):
                zip_path = os.path.join(d, f)
                extract_dir = os.path.join(WORKING, "lora")
                os.makedirs(extract_dir, exist_ok=True)
                with zipfile.ZipFile(zip_path, "r") as z:
                    z.extractall(extract_dir)
                for root, _, files in os.walk(extract_dir):
                    if "adapter_config.json" in files and "adapter_model.safetensors" in files:
                        adapter_path = root
                        break
                if adapter_path:
                    break
        if adapter_path:
            break

if not adapter_path or not has_adapter(adapter_path):
    print("\nYour dataset must contain adapter_config.json and adapter_model.safetensors")
    print("Create new dataset: New Dataset -> Upload -> select payphone-storyteller-lora folder or zip")
    print("Then Add Data to this notebook and re-run.")
    raise FileNotFoundError("LoRA adapter not found.")
print(f"Adapter at: {adapter_path}")

## 3. Merge LoRA → Hugging Face checkpoint + zip


In [ ]:
import gc
import json
import os
import shutil
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModelForCausalLM
import torch

WORKING = globals().get("WORKING", "/kaggle/working")
BASE = "Qwen/Qwen2.5-7B-Instruct"
MERGED_DIR = os.path.join(WORKING, "merged_payphone")
OFFLOAD_DIR = os.path.join(WORKING, "offload")
os.makedirs(MERGED_DIR, exist_ok=True)
os.makedirs(OFFLOAD_DIR, exist_ok=True)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True,
)
print("Loading base model (4-bit, GPU+CPU offload)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE,
    quantization_config=bnb_config,
    device_map="auto",
    max_memory={0: "14GiB", "cpu": "60GiB"},
    offload_folder=OFFLOAD_DIR,
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)

print("Loading LoRA adapter...")
model = PeftModelForCausalLM.from_pretrained(model, adapter_path)

print("Merging...")
model = model.merge_and_unload()

# merge_and_unload from a 4-bit base can leave NF4 Linear layers; save_pretrained then writes
# *.weight.absmax tensors that llama.cpp convert_hf_to_gguf cannot map. Dequantize to FP16 first.
if getattr(model, "is_loaded_in_4bit", False) and hasattr(model, "dequantize"):
    print("Dequantizing merged weights to FP16 for HF/GGUF export...")
    model = model.dequantize()

# Strip BnB fields so save + future GGUF converters see a plain HF checkpoint (use delattr, not None).
for _attr in ("quantization_config", "pre_quantization_dtype", "_pre_quantization_dtype"):
    if hasattr(model.config, _attr):
        delattr(model.config, _attr)

# Some merges leave dtype objects inside config that are not JSON serializable on save_pretrained.
def _sanitize_jsonable(x):
    if isinstance(x, torch.dtype):
        return str(x).replace("torch.", "")
    if type(x).__name__ == "dtype":  # numpy dtype
        return str(x)
    if isinstance(x, dict):
        return {k: _sanitize_jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_sanitize_jsonable(v) for v in x]
    return x

for _k, _v in list(model.config.__dict__.items()):
    model.config.__dict__[_k] = _sanitize_jsonable(_v)

print("Saving merged model (HF format)...")
model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

_cfg_path = os.path.join(MERGED_DIR, "config.json")
with open(_cfg_path, "r", encoding="utf-8") as f:
    _cfg = json.load(f)
for k in ("quantization_config", "pre_quantization_dtype", "_pre_quantization_dtype"):
    _cfg.pop(k, None)
with open(_cfg_path, "w", encoding="utf-8") as f:
    json.dump(_cfg, f, indent=2, ensure_ascii=False)

del model
gc.collect()
torch.cuda.empty_cache()

zip_base = os.path.join(WORKING, "payphone-merged-hf")
archive_path = shutil.make_archive(zip_base, "zip", root_dir=MERGED_DIR)
print("Merged checkpoint dir:", MERGED_DIR)
print("Zip for download:", archive_path)
print("Next: run the download cell, then GGUF via Colab import notebook or local llama.cpp.")


## 4. Download zip


In [ ]:
import os
from IPython.display import FileLink, display

WORKING = "/kaggle/working"
z = os.path.join(WORKING, "payphone-merged-hf.zip")
if os.path.isfile(z):
    os.chdir(WORKING)
    print("Download merged Hugging Face checkpoint (unzip → run convert_hf_to_gguf locally; or use Colab with your LoRA zip for GGUF):")
    display(FileLink("payphone-merged-hf.zip", result_html_prefix="Download: "))
else:
    print("ERROR: payphone-merged-hf.zip not found. Re-run the merge cell.")


## Optional: Convert merged HF → GGUF

**On Kaggle:** use [kaggle_hf_to_gguf.ipynb](https://github.com/s4ifk99/ProjectPayphone/blob/main/notebooks/kaggle_hf_to_gguf.ipynb) — **Add data** your `payphone-merged-hf.zip`, **Internet** ON, run all cells → download `payphone-story.gguf`.

**Colab:** [colab_full_import.ipynb](https://colab.research.google.com/github/s4ifk99/ProjectPayphone/blob/main/notebooks/colab_full_import.ipynb) with **`payphone-storyteller-lora.zip`** does merge + GGUF in one run (adapter zip, not the merged HF zip).

**Local (with llama.cpp cloned):**

```bash
python llama.cpp/convert_hf_to_gguf.py /path/to/merged_payphone --outfile payphone-story.gguf --outtype q8_0
```

Or [scripts/convert_to_gguf.sh](https://github.com/s4ifk99/ProjectPayphone/blob/main/scripts/convert_to_gguf.sh) with `LLAMA_CPP_PATH` and merged dir.
